In [5]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb
from openai import OpenAI
import warnings
warnings.filterwarnings('ignore')

load_dotenv('../.env')
api_key = os.getenv('OPENAI_API_KEY')
print("API key loaded:", "✅" if api_key else "❌ NOT FOUND")

API key loaded: ❌ NOT FOUND


In [6]:
chroma_client = chromadb.PersistentClient(path='../vector_store/chroma_db')
collection = chroma_client.get_collection("complaints")
print("ChromaDB docs:", collection.count())

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

ChromaDB docs: 59766


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [7]:
def retrieve_chunks(query, n_results=5, product_filter=None):
    query_embedding = model.encode([query]).tolist()
    where_filter = {"product": product_filter} if product_filter else None
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where_filter,
        include=['documents', 'metadatas', 'distances']
    )
    
    chunks = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        chunks.append({
            'text': doc,
            'product': meta['product'],
            'relevance_score': round(1 - dist, 4)
        })
    
    return chunks

# Test
test_chunks = retrieve_chunks("unauthorized charge on my credit card")
print(f"Retrieved {len(test_chunks)} chunks\n")
for c in test_chunks:
    print(f"[{c['product']}] Score: {c['relevance_score']}")
    print(c['text'][:150])
    print()

Retrieved 5 chunks

[Credit Cards] Score: 0.7744
2019 I locked the card and announced that further charges are unauthorized. I called several times and contacted via web chat. I even cancelled one ca

[Savings Accounts] Score: 0.7584
I woke up Saturday  to an unauthorized charge by for 180.00. I complained to my bank  Chine but they said they couldn't dispute it until it wasn't pen

[Credit Cards] Score: 0.7483
In the amount of 3500.00 a unauthorized charge was made to my card that I wasnt aware of until I got the statement, discovered said they would look in

[Credit Cards] Score: 0.7376
. I reported that I did not authorize the charge. My credit card has a chip embedded and the charge was not process by the chip, the magnetic swipe no

[Credit Cards] Score: 0.7369
Card charges unmade by me and was charged for items that I did not buy. Every time I would make a payment on my credit card they would take off those 



In [8]:
from groq import Groq
from dotenv import load_dotenv
import os

# Force reload .env
load_dotenv('../.env', override=True)

groq_key = os.getenv('GROQ_API_KEY')
print("Groq key loaded:", "✅" if groq_key else "❌ NOT FOUND")

groq_client = Groq(api_key=groq_key)

def generate_answer(query, chunks):
    context = ""
    for i, chunk in enumerate(chunks, 1):
        context += f"\n[Complaint {i} — {chunk['product']}]\n{chunk['text']}\n"
    
    prompt = f"""You are a financial analyst assistant for CrediTrust Financial.
Your job is to answer questions about customer complaints using only the provided complaint excerpts.

COMPLAINT EXCERPTS:
{context}

QUESTION: {query}

Instructions:
- Answer based only on the complaint excerpts above
- Be specific and cite patterns you see across complaints
- If the excerpts don't contain enough info, say so clearly
- Keep your answer concise (3-5 sentences)

ANSWER:"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.3
    )
    
    return response.choices[0].message.content

# Test it
answer = generate_answer("unauthorized charge on my credit card", test_chunks)
print("ANSWER:\n", answer)

Groq key loaded: ✅
ANSWER:
 Several complaints mention experiencing unauthorized charges on their credit cards, with amounts ranging from unknown to $3500.00 (Complaint 3). A pattern seen across complaints is the difficulty in getting these charges removed, despite reporting them to the credit card company (Complaints 1, 3, and 4). In some cases, the credit card company's response is inadequate, with the customer being sent bills for unauthorized charges (Complaint 1) or the charge being processed without proper verification (Complaint 4). The excerpts suggest a need for more effective resolution of unauthorized charge disputes.


In [9]:
def rag_pipeline(query, n_results=5, product_filter=None):
    print(f"Question: {query}\n")
    
    chunks = retrieve_chunks(query, n_results=n_results, product_filter=product_filter)
    print(f"Retrieved {len(chunks)} chunks:")
    for c in chunks:
        print(f"  [{c['product']}] relevance: {c['relevance_score']}")
    
    print("\nGenerating answer...")
    answer = generate_answer(query, chunks)
    
    print(f"\n{'='*60}")
    print("ANSWER:")
    print(answer)
    print('='*60)
    
    return {"query": query, "chunks": chunks, "answer": answer}

In [10]:
test_questions = [
    "What are the most common issues customers face with credit cards?",
    "Why do customers complain about money transfers?",
    "What problems do people report with their savings accounts?",
    "How do customers describe unauthorized transactions?",
    "What are common complaints about personal loans?"
]

results = []
for q in test_questions:
    print("\n" + "="*70)
    result = rag_pipeline(q)
    results.append(result)
    print()


Question: What are the most common issues customers face with credit cards?

Retrieved 5 chunks:
  [Credit Cards] relevance: 0.6764
  [Credit Cards] relevance: 0.6283
  [Credit Cards] relevance: 0.6154
  [Credit Cards] relevance: 0.6113
  [Credit Cards] relevance: 0.6102

Generating answer...

ANSWER:
Based on the complaint excerpts, the most common issues customers face with credit cards appear to be related to poor customer service and unfair treatment. Multiple complaints (Complaint 1, Complaint 2, and Complaint 5) mention frustrating experiences, unprofessional behavior, and a sense of being treated unfairly. Complaint 2 specifically mentions "abusive discrimination" and "worst customer service," while Complaint 5 states that credit card companies "act unprofessional and play with the financial situation" of customers. However, the excerpts do not provide enough information to identify specific patterns or issues beyond general dissatisfaction with customer service.


Question: Wh

In [11]:
eval_df = pd.DataFrame([{
    'question': r['query'],
    'answer': r['answer'],
    'num_chunks_retrieved': len(r['chunks']),
    'avg_relevance': round(np.mean([c['relevance_score'] for c in r['chunks']]), 4)
} for r in results])

eval_df.to_csv('../data/processed/rag_evaluation.csv', index=False)
print("Evaluation saved!")
eval_df[['question', 'avg_relevance']]

Evaluation saved!


,question,avg_relevance
0,What are the most common issues customers face...,0.6283
1,Why do customers complain about money transfers?,0.6048
2,What problems do people report with their savi...,0.5822
3,How do customers describe unauthorized transac...,0.6428
4,What are common complaints about personal loans?,0.6330
